# Scorer pilot (T-025) on Kaggle

Pilot of the scorer architecture proposed in `docs/scorer-survey.md`: frozen encoders (CLIP ViT-L/14, PickScore,
SigLIP, DINOv2) embed every original, edit and request of the ranked v1 samples once; small heads are then trained on
the preference pairs (A > B) and give one score per edit. Compared with zero-shot baselines (CLIP-score, ...) and
the human ceiling, on the fixed dev and test splits (`src/scorer/data.py`).

Before running:
- input: the private dataset `editjudge-data` (with `ranking/v1/final/`);
- settings: GPU T4 (one is enough), internet on;
- the code is cloned from GitHub (`BRANCH`), so push your changes first;
- choose `STEP` below and get the coordinator's approval for the run.

| STEP | what | time (estimate) |
|---|---|---|
| `smoke` | CLIP ViT-L/14 on 50 samples, baselines, one head | 5 min + model download |
| `full` | 4 backbones on all ranked samples (about 24,000 images), baselines, all heads of `configs/scorer.toml` | 40-60 min |
| `train` | heads and report only, from the embeddings of an earlier session (`PREVIOUS`) | 5-10 min, no GPU needed |

Outputs in `/kaggle/working/out/scorer/`: `report.md` (the comparison table), `baselines.json`,
`runs/<name>/metrics.json`, `embeddings/<backbone>/meta.json` (time per image). Bring back `report.md`,
`baselines.json`, `runs/*/metrics.json` and the `meta.json` files; the embeddings and `scores.jsonl` stay on Kaggle
(derived from team data, never committed).

In [ ]:
import os, shutil, subprocess, sys

REPO = "https://github.com/CryAndRRich/edit-judge.git"
BRANCH = "scorer/trung"                    # the T-025 branch
DATA = "/kaggle/input/editjudge-data"      # the private dataset (editjudge-data.zip, extracted by Kaggle)
CODE = "/kaggle/working/edit-judge"
OUT = "/kaggle/working/out"

if os.path.exists(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

os.environ["EJ_DATA_DIR"] = DATA           # read-only input
os.environ["EJ_OUTPUT_DIR"] = OUT          # all results are written here
os.makedirs(OUT, exist_ok=True)
os.chdir(CODE)

# Model weights go to /tmp: /kaggle/working is limited to 20 GB and is saved as the notebook's output
os.environ["HF_HOME"] = "/tmp/hf"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
try:                                       # optional Kaggle secret HF_TOKEN (Add-ons > Secrets)
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:                          # public models need no token: hide the rate-limit notice
    os.environ["HF_HUB_VERBOSITY"] = "error"

PREVIOUS = None                            # e.g. "/kaggle/input/<this notebook's previous version>/out"
if PREVIOUS:
    shutil.copytree(f"{PREVIOUS}/scorer/embeddings", f"{OUT}/scorer/embeddings", dirs_exist_ok=True)

In [ ]:
!pip -q install -U "transformers>=4.56" sentencepiece
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__)"
!python -m src.scorer list

In [ ]:
STEP = "smoke"          # smoke | full | train
BACKBONES = ["clip-l14", "pickscore", "siglip", "dinov2"]

def ej(*args):
    """python -m src.scorer <args>, printing its output."""
    subprocess.run([sys.executable, "-m", "src.scorer", *map(str, args)], check=True)

if STEP == "smoke":
    ej("embed", "--backbone", "clip-l14", "--limit", 50)
    ej("baselines")
    ej("train", "--run", "mlp-clip-soft")
elif STEP == "full":
    for b in BACKBONES:
        ej("embed", "--backbone", b)
    ej("baselines")
    ej("train")
elif STEP == "train":
    ej("baselines")
    ej("train")
ej("report")

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT}/scorer/report.md").read()))

In [ ]:
# Small files to bring back (the embeddings and per-edit scores stay on Kaggle)
import glob, json, zipfile
keep = [f"{OUT}/scorer/{p}" for p in ("report.md", "report.json", "baselines.json")]
keep += glob.glob(f"{OUT}/scorer/runs/*/metrics.json") + glob.glob(f"{OUT}/scorer/embeddings/*/meta.json")
with zipfile.ZipFile(f"{OUT}/scorer_results.zip", "w") as zf:
    for f in keep:
        if os.path.exists(f):
            zf.write(f, os.path.relpath(f, OUT))
print(f"{OUT}/scorer_results.zip:", len(keep), "files")
for f in glob.glob(f"{OUT}/scorer/embeddings/*/meta.json"):
    print(json.load(open(f)))